# Wizualna Analiza Procesów DSP w Pipeline EEG
Ten notebook stanowi empiryczne potwierdzenie założeń teoretycznych zawartych w analizie technicznej. Prześledzimy transformację sygnału od surowego zapisu do wygładzonego spektrogramu 128x128.

---

In [ ]:
import mne
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import savgol_filter
from scipy.ndimage import gaussian_filter
import seaborn as sns

sns.set_theme(style="whitegrid")
EDF_PATH = 'dane/Alpha1_raw.edf'

# Załadowanie danych do wizualizacji
raw = mne.io.read_raw_edf(EDF_PATH, preload=True, verbose=False)
mapping = {ch: ch.split('-')[0].split(':')[0].strip() for ch in raw.ch_names}
raw.rename_channels(mapping)
raw.set_montage('standard_1020', on_missing='ignore')
raw.pick_types(eeg=True)

print("Sygnał załadowany. Rozpoczynamy analizę procesów.")

## 1. Kondycjonowanie i CAR (Common Average Reference)
**Dlaczego tak:** Przed wejściem do modeli zaawansowanych, sygnał jest początkowo przepuszczany przez filtry pasmowo-przepustowe (1-40 Hz) oraz filtry zaporowe (Notch 50 Hz), aby pozbyć się przydźwięku sieciowego i dryftów wysokoczęstotliwościowych. Następnie aplikowana jest metoda Common Average Reference (CAR). Jest to operacja liniowa, podczas której dla każdej unikalnej próbki czasowej obliczana jest średnia matematyczna potencjału odczytywanego ze wszystkich elektrod. Wyliczona średnia jest następnie odejmowana od napięcia każdego poszczególnego kanału. Podejście to pozwala wyeliminować "szum wspólny" obecny w całym układzie oraz uniezależnia badanie od potencjalnego zakłócenia pochodzącego z miejsca wpięcia fizycznej elektrody referencyjnej.
**Wizualizacja:** Porównanie sygnału przed i po zastosowaniu CAR.

In [ ]:
# Przed filtrowaniem i CAR (surowy, ale po mapowaniu)
data_orig = raw.get_data(picks=['Fp1'])

raw_filtered = raw.copy().filter(1.0, 40.0, verbose=False)
raw_car = raw_filtered.copy().set_eeg_reference('average', projection=False, verbose=False)

plt.figure(figsize=(15, 6))
plt.plot(raw.times[:1000], data_orig[0, :1000] * 1e6, label='Surowy (z referencją fizyczną)', alpha=0.5)
plt.plot(raw.times[:1000], raw_car.get_data(picks=['Fp1'])[0, :1000] * 1e6, label='Po Filtracji + CAR', color='green')
plt.title("Wpływ CAR na stabilność linii bazowej (Kanał Fp1)")
plt.ylabel("Amplituda [uV]")
plt.legend()
plt.show()

## 2. Ślepa separacja źródeł (ICA)
**Dlaczego tak:** W celu usunięcia bardzo silnych i trudnych do zniwelowania filtrami artefaktów (zwłaszcza EOG – artefaktów ocznych, takich jak mrugnięcia) implementowana jest dekompozycja ICA (w wariancie FastICA, z wyodrębnieniem 15 składowych). ICA jest metodą, która z założenia traktuje sygnał zbiorczy EEG z elektrod jako sumę liniową statystycznie niezależnych, nienormalnych (nie-gaussowskich) źródeł generujących potencjał. Algorytm iteracyjnie odwraca macierz mieszającą i dąży do maksymalizacji wariancji lub nienormalności poszczególnych "składowych". Następnie system wylicza korelacje wyizolowanych komponentów ze wskaźnikami aktywności z kanałów czołowych (np. Fp1 i Fp2). Zidentyfikowane tą drogą składowe z mrugnięć są trwale wyzerowane, po czym czysty sygnał jest transformowany i rzutowany z powrotem na przestrzeń czasową.
**Wizualizacja:** Komponenty ICA oraz sygnał po usunięciu mrugnięć.

In [ ]:
ica = mne.preprocessing.ICA(n_components=15, random_state=97, verbose=False)
ica.fit(raw_car)

eog_indices, _ = ica.find_bads_eog(raw_car, ch_name=['Fp1', 'Fp2'], verbose=False)
ica.exclude = eog_indices

raw_clean = raw_car.copy()
ica.apply(raw_clean, verbose=False)

print(f"Zidentyfikowano komponenty mrugnięć: {ica.exclude}")
ica.plot_components(picks=ica.exclude, title="Komponenty zidentyfikowane jako mrugnięcia");

## 3. Nieliniowe Wygładzanie (Savitzky-Golay)
**Dlaczego tak:** Wyizolowany i zrekonstruowany z ICA sygnał czasowy zostaje poddany operacji nieliniowego szlifowania za pomocą filtru Savitzky'ego-Golaya. Standardowe uśrednianie w przesuwnym oknie (SMA) mogłoby zniekształcić morfologię fali EEG. Ten filtr natomiast dopasowuje, metodą najmniejszych kwadratów, wielomian niskiego stopnia w zadanym lokalnym oknie, które przesuwa się wzdłuż osi czasu. Narzędzie to efektywnie filtruje wysoko-częstotliwościowe wibracje, pozostawiając strome i bardzo gwałtowne fale (na przykład iglice epileptoidalne lub ujemne załamki kognitywne), nie tracąc przy tym ich amplitudy ani zjawisk fazowych.
**Wizualizacja:** Zbliżenie na sygnał - redukcja mikro-szumów przy zachowaniu fazy.

In [ ]:
data_clean = raw_clean.get_data(picks=['Fp1'])
data_savgol = savgol_filter(data_clean, window_length=15, polyorder=2, axis=1)

plt.figure(figsize=(15, 6))
plt.plot(raw_clean.times[200:400], data_clean[0, 200:400] * 1e6, 'o-', label='Po ICA (Ząbkowany)', alpha=0.4, markersize=3)
plt.plot(raw_clean.times[200:400], data_savgol[0, 200:400] * 1e6, 'r-', label='Po Savitzky-Golay (Gładki)')
plt.title("Precyzja Savitzky-Golay: Wygładzanie mikro-oscylacji")
plt.legend()
plt.show()

## 4. Estymacja PSD Metodą Welcha
**Dlaczego tak:** Sygnał ulega podziałowi na 2-sekundowe epoki czasowe. Zastosowanie pojedynczej transformaty Fouriera (FFT) na całym segmencie zaowocowałoby widmem charakteryzującym się wysoką wariancją (silnie "zaszumiony" wykres PSD). Stąd aplikowana jest estymacja Welcha (wyliczająca PSD – Power Spectral Density). Podstawą tej modyfikacji jest to, że pojedyncza epoka dzielona jest na krótsze, w dużym stopniu nakładające się okna podrzędne, aplikuje się do nich funkcję okna zapobiegającą wyciekom widma, wylicza transformaty dla każdego podrzędnego, po czym wyniki wszystkich modyfikowanych periodogramów są na koniec uśredniane. Efekt końcowy cechuje znacznie stabilniejsza reprezentacja i wygładzenie szumów dziedziny częstotliwości (kosztem lekkiego spadku precyzji w skali częstotliwości na Hz).
**Wizualizacja:** Porównanie FFT (surowej) z metodą Welcha.

In [ ]:
epochs = mne.make_fixed_length_epochs(raw_clean, duration=2.0, preload=True, verbose=False)

# Welch
spectrum_welch = epochs[0].compute_psd(method='welch', n_fft=256, verbose=False)
psd_welch = spectrum_welch.get_data()[0, 0] # Kanał 0
freqs = spectrum_welch.freqs

plt.figure(figsize=(12, 6))
plt.semilogy(freqs, psd_welch, label='Metoda Welcha (Uśredniona)', color='blue', linewidth=2)
plt.title("Stabilność Widmowa: Metoda Welcha")
plt.xlabel("Częstotliwość [Hz]")
plt.ylabel("Moc [V^2/Hz]")
plt.legend()
plt.show()

## 5. Splot Przestrzenny (Gauss) na Obrazie 128x128
**Dlaczego tak:** Dane z PSD po przekształceniu na skalę logarytmiczną oraz przeskalowaniu min-max interpretowane są jako obraz – dwuwymiarowy spektrogram o wielkości 128x128 punktów. Na końcu traktuje się je filtrem Gaussa, który pod kątem operacyjnym jest matematycznym splotem dwuwymiarowym (konwolucją) całego wygenerowanego obrazu poprzez jądro z rozkładem wariancji opisanym funkcją dzwonową Gaussa. W domenie obrazu działa to jak dwuwymiarowy filtr dolnoprzepustowy usuwając "ziarnistość", tak by sieć konwolucyjna VQ-VAE analizowała wyłącznie wielkoskalowe, wyraźne motywy korelacyjne, a nie szukała nieistotnych wzorców i wariancji w artefaktowym, poszarpanym błędzie estymacji.
**Wizualizacja:** Mapa 2D spektrogramu przed i po konwolucji Gaussa.

In [ ]:
# Przygotowanie macierzy (kanały x częstotliwości)
img_raw = np.log10(spectrum_welch.get_data()[0] + 1e-12)
img_norm = (img_raw - img_raw.min()) / (img_raw.max() - img_raw.min())

# Konwolucja Gaussa (Splot)
img_gauss = gaussian_filter(img_norm, sigma=0.8)

fig, ax = plt.subplots(1, 2, figsize=(16, 8))
sns.heatmap(img_norm, ax=ax[0], cbar=False, cmap='magma')
ax[0].set_title("Spektrogram 128x128 (Przed splotem)")

sns.heatmap(img_gauss, ax=ax[1], cbar=False, cmap='magma')
ax[1].set_title("Spektrogram 128x128 (Po splocie Gaussa)")
plt.tight_layout()
plt.show()

---
### Wnioski
Jak widać na powyższych wizualizacjach, każdy krok pipeline'u pełni rolę filtru statystycznego. 
1. **CAR** stabilizuje tło.
2. **ICA** usuwa dominujące artefakty fizjologiczne.
3. **Savitzky-Golay** wygładza lokalny szum czasowy.
4. **Welch** tworzy stabilną bazę częstotliwościową.
5. **Gauss** przygotowuje spójną teksturę dla warstw konwolucyjnych modelu **AtomVQVAE**.